In [1]:
import yfinance as yf
import pandas as pd
import os
import time

# Liste des actions par secteur (Correction de 'BRK-B' en 'BRK.B')
STOCKS = {
    'Technology': ['AAPL', 'ADBE', 'GOOGL', 'INTC', 'MSFT', 'NVDA'],
    'Finance': ['BAC', 'GS', 'JPM', 'MA', 'V'],
    'Healthcare': ['JNJ', 'PFE', 'UNH'],
    'Consumer Discretionary': ['AMZN', 'DIS', 'HD', 'TSLA', 'NFLX'],
    'Consumer Staples': ['KO', 'PEP', 'PG'],
    'Communication': ['CMCSA'],
}


def download_stock_data(start_date='2013-01-01', end_date='2023-01-01', interval='1d'):
    """
    Télécharge les données historiques des actions spécifiées.

    Parameters:
    -----------
    start_date : str
        Date de début au format 'YYYY-MM-DD'
    end_date : str
        Date de fin au format 'YYYY-MM-DD'
    interval : str
        Intervalle des données ('1d' pour journalier)

    Returns:
    --------
    dict : Dictionnaire contenant les DataFrames pour chaque action
    """
    stock_dir = 'stock'
    os.makedirs(stock_dir, exist_ok=True)

    stock_data_dir = 'stock_data'
    os.makedirs(stock_data_dir, exist_ok=True)

    all_stocks = [stock for sector in STOCKS.values() for stock in sector]
    stock_data = {}

    for stock in all_stocks:
        try:
            print(f"Téléchargement des données pour {stock}...")

            # Utilisation de yf.download() au lieu de yf.Ticker().history()
            df = yf.download(stock, start=start_date, end=end_date, interval=interval)

            if df.empty:
                print(f"Attention: Aucune donnée trouvée pour {stock}")
                continue

            df.index = pd.to_datetime(df.index)
            # Correction pour gérer le MultiIndex des colonnes
            #df.columns = [''.join(col).upper() for col in df.columns]
            df.columns = ["Close", "High", "Low", "Open", "Volume"]
            csv_path = os.path.join(stock_dir, f'{stock}.csv')
            df.to_csv(csv_path)
            print(f"Données sauvegardées dans {csv_path}")

            stock_data[stock] = df

            # Pause pour éviter d’être bloqué par Yahoo Finance
            time.sleep(2)

        except Exception as e:
            print(f"Erreur lors du téléchargement de {stock}: {str(e)}")

    summary = []
    for stock, data in stock_data.items():
        summary.append({
            'Stock': stock,
            'Start Date': data.index.min(),
            'End Date': data.index.max(),
            'Trading Days': len(data),
            'Missing Days': data.isnull().sum().sum()
        })

    summary_df = pd.DataFrame(summary)
    summary_path = os.path.join(stock_data_dir, 'download_summary.csv')
    summary_df.to_csv(summary_path, index=False)
    print(f"\nRapport de synthèse sauvegardé dans '{summary_path}'")

    return stock_data

def check_data_quality(stock_data):
    """
    Vérifie la qualité des données téléchargées.

    Parameters:
    -----------
    stock_data : dict
        Dictionnaire contenant les DataFrames des actions
    """
    stock_data_dir = 'stock_data'
    os.makedirs(stock_data_dir, exist_ok=True)

    report = []

    for stock, df in stock_data.items():
        missing = df.isnull().sum()
        dates = df.index
        date_gaps = [
            f"{dates[i-1].date()} to {dates[i].date()}"
            for i in range(1, len(dates)) if (dates[i] - dates[i-1]).days > 1
        ]

        report.append({
            'Stock': stock,
            'Missing Values': missing.to_dict(),
            'Date Gaps': date_gaps,
            'Total Days': len(df)
        })

    report_path = os.path.join(stock_data_dir, 'quality_report.txt')
    with open(report_path, 'w') as f:
        for r in report:
            f.write(f"\nRapport pour {r['Stock']}:\n")
            f.write(f"Nombre total de jours: {r['Total Days']}\n")
            f.write("Valeurs manquantes:\n")
            for col, count in r['Missing Values'].items():
                if count > 0:
                    f.write(f"  {col}: {count}\n")
            if r['Date Gaps']:
                f.write("Gaps dans les dates:\n")
                for gap in r['Date Gaps']:
                    f.write(f"  {gap}\n")
            f.write("-" * 50 + "\n")

    print(f"Rapport de qualité sauvegardé dans '{report_path}'")

if __name__ == "__main__":
    stock_data = download_stock_data()
    check_data_quality(stock_data)

Téléchargement des données pour AAPL...
YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


Données sauvegardées dans stock\AAPL.csv
Téléchargement des données pour ADBE...


[*********************100%***********************]  1 of 1 completed


Données sauvegardées dans stock\ADBE.csv
Téléchargement des données pour GOOGL...


[*********************100%***********************]  1 of 1 completed


... (37 outputs left out)

[*********************100%***********************]  1 of 1 completed


Données sauvegardées dans stock\PG.csv
Téléchargement des données pour CMCSA...


[*********************100%***********************]  1 of 1 completed


Données sauvegardées dans stock\CMCSA.csv

Rapport de synthèse sauvegardé dans 'stock_data\download_summary.csv'
Rapport de qualité sauvegardé dans 'stock_data\quality_report.txt'


In [2]:
import os
import pandas as pd

def combine_adj_close(stock_dir='stock'):
    # Créer une liste pour stocker les DataFrames des différentes actions
    adj_close_data = []

    # Parcourir tous les fichiers CSV du dossier
    for filename in os.listdir(stock_dir):
        if filename.endswith('.csv'):
            # Charger le fichier CSV dans une DataFrame
            csv_path = os.path.join(stock_dir, filename)
            try:
                data = pd.read_csv(csv_path, index_col=0, parse_dates=True)

                # Vérifier si la colonne 'Adj Close' existe
                if 'Close' in data.columns:
                    # Extraire la colonne 'Adj Close' et l'ajouter à la liste
                    adj_close_data.append(data[['Close']].rename(columns={'Close': filename.split('.')[0]}))
                else:
                    print(f"Colonne 'close' absente dans {filename}.")
            except Exception as e:
                print(f"Erreur lors du traitement de {filename}: {str(e)}")

    # Combiner toutes les DataFrames en une seule
    if adj_close_data:
        combined_data = pd.concat(adj_close_data, axis=1)
        return combined_data
    else:
        print("Aucune donnée n'a été trouvée.")
        return None
    return adj_close_data

# Exemple d'utilisation :
combined_df = combine_adj_close(stock_dir='stock')
if combined_df is not None:
    print(combined_df.head())  # Afficher les premières lignes de la DataFrame combinée


                 AAPL       ADBE     AMZN       BAC      CMCSA        DIS  \
Date                                                                        
2013-01-02  16.669012  38.340000  12.8655  9.606839  14.790046  45.887760   
2013-01-03  16.458618  37.750000  12.9240  9.550938  14.624942  45.986538   
2013-01-04  16.000164  38.130001  12.9575  9.670721  14.617267  46.866585   
2013-01-07  15.906051  37.939999  13.4230  9.654751  14.551998  45.771023   
2013-01-08  15.948846  38.139999  13.3190  9.566910  14.651824  45.582443   

                GOOGL          GS         HD       INTC  ...         MA  \
Date                                                     ...              
2013-01-02  18.013227  104.753006  47.893547  15.289290  ...  47.387516   
2013-01-03  18.023689  104.180168  47.757751  15.246385  ...  47.455284   
2013-01-04  18.379845  107.020546  47.667217  15.131967  ...  47.453438   
2013-01-07  18.299648  106.821625  47.410694  15.196328  ...  48.275803   
2013-01-08

In [3]:
combined_df.to_csv("df_historiques.csv")